# Laboratorio 3.4 — Entrenamiento de un modelo

**Estudiante: Santiago Perez Garzon**

**Ejecución:** consultar la verificación al final del notebook. Antes de ejecutar, verificar que el bucket y el rol corresponden al laboratorio activo.

Se sigue el flujo oficial de AWS Academy en Amazon SageMaker: preparación, división de datos, almacenamiento en S3 y entrenamiento con XGBoost.

**Uso ético de IA:** la IA asistió en la revisión del código, la preparación del notebook y la documentación. Los resultados deben provenir de la ejecución real en AWS y ser revisados por el estudiante.

# Lab 3.4 - Student Notebook

## Overview

This lab is a continuation of the guided labs in Module 3. 

In this lab, you will split the data into three separate datasets:

- *Training Set* - This will be used to train the model.
- *Validation Set* - This will be used during training to validate the model.
- *Test Set* - This will be held back and used to produce metrics after the model is trained. You will use this dataset in an upcoming lab.

With the split data, you will train a XGBoost model by using Amazon SageMaker.


## Introduction to the business scenario

You work for a healthcare provider, and want to improve detection of abnormalities in orthopedic patients. 

You are tasked with solving this problem by using machine learning (ML). You have access to a dataset that contains six biomechanical features and a target of *normal* or *abnormal*. You can use this dataset to train an ML model to predict if a patient will have an abnormality.


## About this dataset

This biomedical dataset was built by Dr. Henrique da Mota during a medical residence period in the Group of Applied Research in Orthopaedics (GARO) of the Centre Médico-Chirurgical de Réadaptation des Massues, Lyon, France. The data has been organized in two different, but related, classification tasks. 

The first task consists in classifying patients as belonging to one of three categories: 

- *Normal* (100 patients)
- *Disk Hernia* (60 patients)
- *Spondylolisthesis* (150 patients)

For the second task, the categories *Disk Hernia* and *Spondylolisthesis* were merged into a single category that is labeled as *abnormal*. Thus, the second task consists in classifying patients as belonging to one of two categories: *Normal* (100 patients) or *Abnormal* (210 patients).


## Attribute information:

Each patient is represented in the dataset by six biomechanical attributes that are derived from the shape and orientation of the pelvis and lumbar spine (in this order): 

- Pelvic incidence
- Pelvic tilt
- Lumbar lordosis angle
- Sacral slope
- Pelvic radius
- Grade of spondylolisthesis

The following convention is used for the class labels: 
- DH (Disk Hernia)
- Spondylolisthesis (SL)
- Normal (NO) 
- Abnormal (AB)


For more information about this dataset, see the [Vertebral Column dataset webpage](http://archive.ics.uci.edu/ml/datasets/Vertebral+Column).


## Dataset attributions

This dataset was obtained from:
Dua, D. and Graff, C. (2019). UCI Machine Learning Repository (http://archive.ics.uci.edu/ml). Irvine, CA: University of California, School of Information and Computer Science.


# Lab setup
Because this solution is split across several labs in the module, you must run the following cells so that you can load the data.

## Importing the data

By running the following cells, the data will be imported and ready for use. 

**Note:** The following cells represent the key steps in the previous labs.

In [1]:
import warnings, requests, zipfile, io
warnings.simplefilter('ignore')
import pandas as pd
from scipy.io import arff
import boto3

In [2]:
f_zip = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00212/vertebral_column_data.zip'
r = requests.get(f_zip, stream=True, timeout=60)
r.raise_for_status()
Vertebral_zip = zipfile.ZipFile(io.BytesIO(r.content))
Vertebral_zip.extractall()

In [3]:
data = arff.loadarff('column_2C_weka.arff')
df = pd.DataFrame(data[0])

In [4]:
class_mapper = {b'Abnormal':1,b'Normal':0}
df['class']=df['class'].replace(class_mapper)

# Step 1: Exploring the data
You will start with a quick reminder of the data in the dataset.

To get the most out of this lab, carefully read the instructions and code before you run the cells. Take time to experiment!

First, use **shape** to examine the number of rows and columns.

In [5]:
df.shape

(310, 7)

Next, get a list of the columns.

In [6]:
df.columns

Index(['pelvic_incidence', 'pelvic_tilt', 'lumbar_lordosis_angle',
       'sacral_slope', 'pelvic_radius', 'degree_spondylolisthesis', 'class'],
      dtype='object')

You can see the six biomechanical features, and that the target column is named *class*.


# Step 2: Preparing the data

For this lab, you must split the data into three datasets.

An internet search will show many different ways to split datasets. Many code samples that you might find will split the dataset into the *target* and the *features*. Then, they will split each of those two datasets into three subsets, which results in a total of six datasets to track.

## Moving the target column position

XGBoost requires the training data to be in a single file. The file must have the target value be the first column. 

Get the target column and move it to the first position.

In [7]:
cols = df.columns.tolist()
cols = cols[-1:] + cols[:-1]
df = df[cols]

You should see that the **class** is now the first column.

In [8]:
df.columns

Index(['class', 'pelvic_incidence', 'pelvic_tilt', 'lumbar_lordosis_angle',
       'sacral_slope', 'pelvic_radius', 'degree_spondylolisthesis'],
      dtype='object')

## Splitting the data

You will start by splitting the dataset into two datasets. You will use one dataset for training, and you will split the other dataset again for use with validation and testing.

You will use the *train_test_split function* from the *scikit-learn library*, which is a free machine learning library for Python. It has many algorithms and useful functions, such as the one you will use. 

- For more information about the function, see the [Train_test_split documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). 
 - For more information about scikit-learn, see the [scikit-learn guide](https://scikit-learn.org/stable/)

Because you don't have a lot of data, you want to make sure that the split datasets contain a representative amount of each class. Thus, you will use the *stratify* switch. Finally, you will use a random number so that you can repeat the splits.

### División reproducible
La semilla 42 permite repetir la división. La estratificación conserva la proporción de clases: 80 % para entrenamiento, 10 % para validación y 10 % para prueba.

In [9]:
from sklearn.model_selection import train_test_split
train, test_and_validate = train_test_split(df, test_size=0.2, random_state=42, stratify=df['class'])

Next, split the *test_and_validate* dataset into two equal parts.

In [10]:
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['class'])

Examine the three datasets.

In [11]:
print(train.shape)
print(test.shape)
print(validate.shape)

(248, 7)
(31, 7)
(31, 7)


Now, check the distribution of the classes.

In [12]:
print(train['class'].value_counts())
print(test['class'].value_counts())
print(validate['class'].value_counts())

class
1    168
0     80
Name: count, dtype: int64
class
1    21
0    10
Name: count, dtype: int64
class
1    21
0    10
Name: count, dtype: int64


## Uploading the data to Amazon S3

XGboost will load the data for training from Amazon Simple Storage Service (Amazon S3). Thus, you must write the data to a comma-separated values (CSV) file, and then upload the file to Amazon S3.

Start by setting up some variables to the S3 bucket, then create a function to upload the CSV file to Amazon S3. You can reuse this function.

First, explore the function.

Note the following line:

`dataframe.to_csv(csv_buffer, header=False, index=False)`

This line writes the pandas DataFrame (which was passed into the function) into the IO buffer that's named *csv_buffer*. You use a buffer because you don't need to write the file locally.

To stop the column headers from being written out, use `header=False`. To stop the pandas index from being output, use `index=False`.

To write the csv_buffer to Amazon S3 as an object, use the `put` operation on the `object`, which is a property of the `bucket`.


### Datos en S3
Se utiliza el bucket del notebook oficial de la sesión activa. Los CSV no incluyen encabezado ni índice y sitúan la etiqueta en la primera columna. Cada partición usa un prefijo separado.

In [13]:
bucket='c221302a5580007l17556420t1w141374916042-labbucket-fxbih8rkkfof'

prefix='lab3'

train_file='vertebral_train.csv'
test_file='vertebral_test.csv'
validate_file='vertebral_validate.csv'

import os

s3_resource = boto3.Session().resource('s3')
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

Use the function that you created to upload the three datasets.

In [14]:
upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

# Step 3: Training the model

Now that the data in Amazon S3, you can train a model. 

The first step is to get the XGBoost container URI.

In [15]:
import boto3
from sagemaker.image_uris import retrieve
container = retrieve(framework='xgboost', region=boto3.Session().region_name, version='1.0-1')


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


Next, you must set some *hyperparameters* for the model. Because this is the first time you are training the model, you can use some values to get started.

### Configuración de XGBoost
Se conservan 42 rondas, binary:logistic y AUC. La probabilidad corresponde a la clase 1 (Abnormal), según el mapeo del notebook. El AUC evalúa la ordenación de casos positivos y negativos.

In [16]:
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

Use the **estimator** function to set up the model. Here are a few parameters of interest:

- **instance_count** - This defines how many instances will be used for training. You will use *one* instance.
- **instance_type** - This defines the instance type for training. In this case, it's *ml.m4.xlarge*.


In [17]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
s3_output_location=f's3://{bucket}/{prefix}/output/'
xgb_model=sagemaker.estimator.Estimator(
    image_uri=container, role=get_execution_role(),
    instance_type='ml.m5.4xlarge', instance_count=1,
    output_path=s3_output_location, hyperparameters=hyperparams,
    sagemaker_session=sagemaker.Session())

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


The estimator needs *channels* to feed data into the model. For training, the *train_channel* and *validate_channel* will be used.

In [18]:
train_channel=TrainingInput(f's3://{bucket}/{prefix}/train/', content_type='text/csv')
validate_channel=TrainingInput(f's3://{bucket}/{prefix}/validate/', content_type='text/csv')
data_channels={'train':train_channel,'validation':validate_channel}

Running **fit** will train the model.

**Note:** This process can take up to 5 minutes.

### Entrenamiento administrado
SageMaker utiliza los canales de entrenamiento y validación y guarda el artefacto del modelo en S3. El conjunto de prueba queda reservado para la evaluación posterior.

In [19]:
xgb_model.fit(inputs=data_channels, logs=False)

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-10-12-09-19-44-233



2026-10-12 09:19:45 Starting - Starting the training job..
2026-10-12 09:20:00 Starting - Preparing the instances for training....
2026-10-12 09:20:27 Downloading - Downloading input data..
2026-10-12 09:20:42 Downloading - Downloading the training image.....
2026-10-12 09:21:12 Training - Training image download completed. Training in progress.......
2026-10-12 09:21:48 Uploading - Uploading generated training model...
2026-10-12 09:22:05 Completed - Training job completed


After the training is complete, you are ready to test and evaluate the model. However, you will do  testing and validation in later labs.

# Congratulations!

You have completed this lab, and you can now end the lab by following the lab guide instructions.

## Verificación de resultados
La siguiente celda comprueba que el trabajo de entrenamiento esté Completed y conservar sus métricas reales. Las métricas de validación no sustituyen la evaluación sobre prueba.

In [20]:
import json
from pathlib import Path
job=xgb_model.latest_training_job.describe()
assert job['TrainingJobStatus']=='Completed'
resultado={k:job[k] for k in ('TrainingJobName','TrainingJobStatus','FinalMetricDataList','HyperParameters','ResourceConfig','ModelArtifacts')}
resultado['filas']={'train':len(train),'validate':len(validate),'test':len(test)}
resultado['compatibilidad']='SDK 2 instalado en la instancia; Estimator.fit equivale al flujo ModelTrainer.train de SDK 3'
Path('resultado_lab34.json').write_text(json.dumps(resultado,indent=2,default=str))
print(json.dumps(resultado,indent=2,default=str))

{
  "TrainingJobName": "sagemaker-xgboost-2026-10-12-09-19-44-233",
  "TrainingJobStatus": "Completed",
  "FinalMetricDataList": [
    {
      "MetricName": "validation:auc",
      "Value": 0.9666699767112732,
      "Timestamp": "2026-10-12 09:21:39+00:00"
    },
    {
      "MetricName": "train:auc",
      "Value": 1.0,
      "Timestamp": "2026-10-12 09:21:39+00:00"
    }
  ],
  "HyperParameters": {
    "eval_metric": "auc",
    "num_round": "42",
    "objective": "binary:logistic"
  },
  "ResourceConfig": {
    "InstanceType": "ml.m5.4xlarge",
    "InstanceCount": 1,
    "VolumeSizeInGB": 30
  },
  "ModelArtifacts": {
    "S3ModelArtifacts": "s3://c221302a5580007l17556420t1w141374916042-labbucket-fxbih8rkkfof/lab3/output/sagemaker-xgboost-2026-10-12-09-19-44-233/output/model.tar.gz"
  },
  "filas": {
    "train": 248,
    "validate": 31,
    "test": 31
  },
  "compatibilidad": "SDK 2 instalado en la instancia; Estimator.fit equivale al flujo ModelTrainer.train de SDK 3"
}
